# 07b — LangGraph Rewrite: Same Deep Research, Different Runtime

In notebook **07** we built deep research as a hand-rolled multi-agent system: a `Coordinator` Python class, two `Researcher` agents running as FastAPI subprocesses, JSON-RPC A2A calls between them, `asyncio.gather` for parallel fan-out. **It worked.** It also exposed four pain points that LangGraph is built to fix.

**By the end of this notebook you will:**
1. Re-implement the **same** decompose → parallel-research → synthesize pipeline as a `StateGraph`
2. Use the **`Send` API** for dynamic fan-out (sub-question count varies per run)
3. Hit and fix the **#1 LangGraph beginner gotcha** — silent data loss in parallel branches without a reducer
4. Inspect the graph's **state history** for free — preview of time-travel debugging (full coverage in 08)

Lecture references: **S9 §2** (LangGraph mental model), **S9 §3.3** (time travel preview).

**What this notebook does NOT add yet:**
- `SqliteSaver` checkpointer (notebook 08)
- `interrupt()` for HITL (notebook 08)
- Worker pipeline / queue / SSE (notebook 09)

We focus on the **orchestration pattern** itself.


## 1. The four pain points from notebook 07

Be honest about what hurt in the hand-rolled version. These are the same things every team feels when they outgrow `asyncio.gather`:

| Pain | Hand-rolled (07) | LangGraph (07b) |
|---|---|---|
| State is scattered across local vars in `Coordinator.run` | every method passes args by hand | one `TypedDict` carries everything |
| Parallel branch failures cascade messily | one researcher dying corrupts the gather | superstep barrier; failed branches replay in isolation |
| No checkpoint — restart = lose all work | hard exit anywhere = start over | checkpointer adds full resumability (notebook 08) |
| Hand-written `asyncio.gather` for fan-out | works but you write it every time | `Send` API + reducer = production-grade map-reduce |

Lecture S9 calls this *"the four-line argument for LangGraph in production: a state machine, durable across crashes, with HITL gates, fully auditable. Building these four pieces by hand on top of a hand-rolled agent loop is months of work."*

Let's see the same pipeline run on LangGraph.


## 2. Setup

In [ ]:
import asyncio
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
print(f"Using model: {MODEL}")
print("Tavily configured:", bool(os.getenv("TAVILY_API_KEY")))

## 3. The state object — TypedDict + reducers

**State design is the most important architectural decision in a LangGraph system.** Too narrow and nodes pass data out-of-band. Too wide and you have an unmanageable blob.

Our state has four fields and three writer patterns:

In [ ]:
import inspect
from deepbrief.agents.langgraph_brief import ResearchState, Finding

print(inspect.getsource(Finding))
print(inspect.getsource(ResearchState))

The interesting line is:

```python
findings: Annotated[list[Finding], operator.add]
```

Without that `Annotated[..., operator.add]` syntax, the field uses the **default reducer = overwrite**. If three parallel `research` nodes all return `{"findings": [...]}`, only one wins — and **which one wins depends on completion order**. That's non-deterministic silent data loss.

With `operator.add`, the three lists CONCAT. All findings preserved.

**Let me prove the bug exists.**

## 4. The reducer bug, demonstrated

I'm going to define **two** states — one without a reducer (default overwrite), one with — and build a tiny graph where three parallel nodes each contribute a finding.

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

# ─── version A: NO reducer (the bug) ─────────────────────────────────
class StateBuggy(TypedDict):
    items: list[str]   # default reducer = overwrite

def buggy_fanout(s):
    return [Send("worker", {"i": i}) for i in range(3)]

def buggy_worker(payload):
    return {"items": [f"finding_{payload['i']}"]}

g_buggy = StateGraph(StateBuggy)
g_buggy.add_node("start", lambda s: {"items": []})
g_buggy.add_node("worker", buggy_worker)
g_buggy.add_edge(START, "start")
g_buggy.add_conditional_edges("start", buggy_fanout, ["worker"])
g_buggy.add_edge("worker", END)

buggy_app = g_buggy.compile()
buggy_result = await buggy_app.ainvoke({"items": []})
print("WITHOUT reducer:", buggy_result)
print("  → expected 3 items, got", len(buggy_result["items"]))

In [ ]:
# ─── version B: WITH reducer (the fix) ───────────────────────────────
class StateFixed(TypedDict):
    items: Annotated[list[str], operator.add]

g_fixed = StateGraph(StateFixed)
g_fixed.add_node("start", lambda s: {"items": []})
g_fixed.add_node("worker", buggy_worker)
g_fixed.add_edge(START, "start")
g_fixed.add_conditional_edges("start", buggy_fanout, ["worker"])
g_fixed.add_edge("worker", END)

fixed_app = g_fixed.compile()
fixed_result = await fixed_app.ainvoke({"items": []})
print("WITH reducer:   ", fixed_result)
print("  → expected 3 items, got", len(fixed_result["items"]))

**This is the gotcha**. The buggy version loses two of the three findings. The fix is one line — wrap the type with `Annotated[..., reducer]`.

**Senior interview answer** (S9 §2.3): *"In a multi-agent LangGraph system, what happens if two parallel branches both modify the same state field with no reducer? The second write silently clobbers the first. Default reducer is overwrite. Declare `Annotated[list, operator.add]` for accumulators, `add_messages` for conversation history, or a custom function for dedup-aware merge."*


## 5. The nodes

Three nodes. Each is a pure function — read state, return a partial update dict.

Notable: `research_node` doesn't take the full state — it takes a `Send` payload (just one sub-question). That's how dynamic fan-out broadcasts different work to each branch.

In [ ]:
from deepbrief.agents.langgraph_brief import (
    decompose_node, plan_research, research_node, synthesize_node,
)
for fn in [decompose_node, plan_research, research_node, synthesize_node]:
    src = inspect.getsource(fn)
    print("─" * 60)
    print(src.split('\"\"\"')[0].strip() + "\n    ...")

## 6. Build the graph

Wire the four structural primitives:

1. **`add_node`** — register the three pure functions.
2. **`add_edge(START, ...)`** — entry point.
3. **`add_conditional_edges(decompose, plan_research, ["research"])`** — `plan_research` returns `list[Send]`, NOT a string. LangGraph detects this and dispatches each Send as its own parallel branch.
4. **`add_edge("research", "synthesize")`** — static edge. The synthesize barrier waits for ALL parallel research branches to complete.


In [ ]:
from deepbrief.agents.langgraph_brief import build_brief_graph, build_brief_app

graph = build_brief_graph()
print("Nodes:", list(graph.nodes))
print("Static edges:")
for e in graph.edges:
    print(f"  {e[0]} → {e[1]}")

app = build_brief_app()
print("\nGraph compiled.")

## 7. Visualise the graph (optional but worth doing)

LangGraph can render the compiled graph as a Mermaid diagram. Useful for documentation, useful for catching topology bugs — if the diagram doesn't match your mental model, your graph is wrong.

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    # PNG render needs `pyppeteer` or a remote service; fall back to ASCII
    print("PNG render skipped:", e)
    print()
    print(app.get_graph().draw_ascii())

## 8. Run it

Same input as notebook 07. Watch what happens.

(This is a real LLM + web search call — takes 30-60s end to end.)

In [ ]:
TOPIC = "State of the Model Context Protocol (MCP) in 2026"

result = await app.ainvoke({"topic": TOPIC, "sub_questions": [], "findings": [], "draft": None})

print("=== SUB-QUESTIONS ===")
for i, q in enumerate(result["sub_questions"], 1):
    print(f"{i}. {q}")

print(f"\n=== {len(result['findings'])} FINDINGS (one per researcher) ===")
for i, f in enumerate(result["findings"], 1):
    print(f"\n--- finding {i}: {f['question']} ---")
    print(f["summary"][:300] + ("..." if len(f["summary"]) > 300 else ""))
    print("sources:", f["sources"][:2])

print("\n=== SYNTHESIZED BRIEF ===")
print(result["draft"])

**What just happened, in superstep terms** (S9 §2.6):

- **Superstep 1**: `decompose` ran alone. 1 LLM call. Wrote `sub_questions` to state.
- **Superstep 2**: `plan_research` returned N `Send` objects → **N `research` nodes ran in parallel** in the same superstep. Each made its own LLM + tool calls. All N completed before the barrier.
- **Superstep 3**: `synthesize` ran alone, reading the now-aggregated `findings` list. 1 LLM call.

If a researcher had crashed in superstep 2, the others' results would have been persisted as **pending writes** — on retry, only the failed researcher would replay, not all of them. That's the all-or-nothing superstep trap (S9 §2.6) working in your favor when paired with a checkpointer.


## 9. Inspect the execution as a stream of typed events

`app.astream_events()` exposes every node start/end/error as a typed event. This is what you wire into your UI for live progress bars and what notebook 09 will plumb into SSE.

In [ ]:
event_counts = {}
async for ev in app.astream_events(
    {"topic": "What is the A2A protocol?", "sub_questions": [], "findings": [], "draft": None},
    version="v2",
):
    kind = ev["event"]
    event_counts[kind] = event_counts.get(kind, 0) + 1

print("Event types observed in one full run:")
for kind, n in sorted(event_counts.items(), key=lambda x: -x[1]):
    print(f"  {n:3d}  {kind}")

Notice the volume of `on_chat_model_*` events — each LLM call inside each ReAct loop inside each researcher emits start/stream/end events. **You get full LLM-level observability without writing a single trace.append() call** — this is the LangGraph value-add over the hand-rolled S8 loop.


## 10. Preview: state history (full coverage in notebook 08)

Without a checkpointer, `app.get_state_history(config)` returns nothing — there's no durable state. But once we add `SqliteSaver` in notebook 08, the same API gives us **every superstep's state** as an inspectable snapshot.

This is how you answer the senior interview question:

> *"A user reports the agent gave a different answer this time for the same query. Walk me through how you'd reproduce."*

Answer: pull `get_state_history(thread_id)`, find the divergence point — usually a tool call that returned different data — replay or fork from that checkpoint. Without a checkpointer you'd be reading logs and guessing.

We do the full reproduce-and-fork demo in notebook 08 once the checkpointer is wired in.


## 11. Side-by-side: hand-rolled vs LangGraph

Same problem, two implementations. The diff that matters:

| Dimension | `coordinator.py` (notebook 07) | `langgraph_brief.py` (this notebook) |
|---|---|---|
| Lines of code | ~110 + ~100 in `a2a/` server/client | ~150 total |
| Parallel safety | `asyncio.gather` with try/except per task | `operator.add` reducer — declared once |
| State management | passed as method args + local vars | one `TypedDict` |
| Error isolation | a researcher crash poisons `gather` | failed branch replays alone |
| Topology visualization | none (read the code) | `draw_mermaid_png()` for free |
| Event stream | hand-write trace.append() | `astream_events()` for free |
| Add HITL gate | implement an approval table from scratch | add `interrupt()` in one node (notebook 08) |
| Add checkpointing | implement state serializer + restore logic | `SqliteSaver()` one line (notebook 08) |

**The line count is comparable. The capabilities are not.** Hand-rolled stops at "it runs." LangGraph gets resumability, observability, HITL, and time-travel for free as you opt into them.


## 12. When NOT to use LangGraph

Honesty check (S9 §6):

- **Pipeline-shaped task** — same exact steps every time → use a plain Python function with `asyncio.gather`. No `StateGraph`. 
- **Single strong agent suffices** — 3 specialists that are all `gpt-4o-mini` with slightly different prompts ≠ specialization. One Sonnet 4.6 with strict mode + good prompt + cost cap matches that on tightly-coupled tasks.
- **Compliance / regulated** — auditors need determinism. Use a deterministic workflow with one or two clearly-bounded LLM steps.

**Three checks before reaching for orchestration:**
1. Does the task path vary meaningfully by input? If no, **pipeline.**
2. Do you genuinely need multiple specialties? If no, **single agent.**
3. Does input volume × token cost justify the 4-15× multiplier? If no, **single agent.**

All three yes → orchestrate. Any no → simpler.


## 13. Self-check

1. What's the default reducer if you don't annotate a field? What's the consequence in parallel branches?
2. When do you use static edges vs the `Send` API for fan-out?
3. A researcher crashes mid-superstep. What does LangGraph preserve, and what gets re-run on retry?
4. We did NOT use a checkpointer in this notebook. What capability do we *not* have? (Hint: notebook 08.)
5. Compare line counts: hand-rolled 07 vs LangGraph 07b. What's the *real* difference?

## What's next

Notebook **08 capstone extension** — add `SqliteSaver` + `interrupt()` + `get_state_history`:
- The brief now pauses for human approval via `interrupt()` (durable HITL, replaces `ApprovalStore`).
- Editor can close their browser, come back tomorrow, resume.
- We demonstrate time-travel by forking from a prior checkpoint with modified sub-questions.

Notebook **09 production pipeline** — wrap the LangGraph app in Redis Streams + FastAPI SSE:
- FastAPI POST `/runs` enqueues a task.
- Worker consumes, acquires SETNX lock, runs the graph, streams progress, stores result.
- This is S9 Part 5 — the lecture maps line-by-line to the code.
